# 01a · 마르코프 사슬의 정의와 채프먼-콜모고로프 방정식 (Markov Chains: Definition and the Chapman-Kolmogorov Equation)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §1.1 (definitions, gambler's ruin·weather chain), §1.2 (Theorem 1.1 = 채프먼-콜모고로프) · Ross 12e §4.1, §4.2 (비 오는 날 사슬의 $P^{(4)}$) · Lawler 2e §1.1 (유한 사슬의 행렬 표기) · Norris §1.1 (Theorem 1.1.1 경로 확률) |
| 선수 노트북 | 00a (랜덤 워크·파산 확률), 00c (조건부 기댓값·전확률 공식) |
| 예상 소요 | 90분 |
| 상태 | draft |

이 노트북의 단 하나의 아이디어: **한 걸음의 규칙이 행렬 $P$이면 $n$걸음의 규칙은 행렬 거듭제곱 $P^n$이다.** 그 이유가 채프먼-콜모고로프 방정식이고, 00a에서 시뮬레이션과 차분방정식으로 얻었던 파산확률을 이번에는 $P^{300}$의 한 원소로 다시 읽어 낸다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from numpy.linalg import matrix_power
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import is_stochastic, simulate_chain, gamblers_ruin_matrix, gamblers_ruin_prob
from spkit.plots import plot_paths

SEED, rng = rng_for("01a")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요. 이번 노트북의 증명은 전부 **전확률 공식**과 **조건부확률의 곱셈 법칙**만으로 이루어진다.

**Q1.** (00c) $E[X\mid Y]$가 '확률변수'라는 말의 뜻은? 그리고 tower property를 식으로 쓰면?

<details><summary>정답</summary>

$E[X\mid Y]=g(Y)$, 즉 $Y$의 함수인 확률변수이다. Tower property: $E\big[E[X\mid Y]\big]=E[X]$. 사건 $A$에 대해 $X=\mathbf 1_A$를 넣으면 전확률 공식 $P(A)=\sum_y P(Y=y)P(A\mid Y=y)$가 된다 — 이 노트북의 모든 증명이 이 한 줄이다.

</details>

**Q2.** (00c) $N\sim\text{Poisson}(\lambda)$, $Y_i$ i.i.d.이고 $N$과 독립일 때 $S=\sum_{i=1}^N Y_i$의 분산은?

<details><summary>정답</summary>

$\operatorname{Var} S=\lambda E[Y^2]$ (복합 푸아송 공식). 일반 랜덤 합은 $\operatorname{Var}S=E[N]\operatorname{Var}Y+\operatorname{Var}N\,(EY)^2$이고, 푸아송은 $E[N]=\operatorname{Var}N=\lambda$라 $\lambda(\operatorname{Var}Y+(EY)^2)=\lambda E[Y^2]$.

</details>

**Q3.** (00c) 독립인 $X_i\sim\text{Exp}(\lambda_i)$, $i=1,\dots,k$에 대해 $P(\min_j X_j = X_i)$는?

<details><summary>정답</summary>

$\lambda_i/\sum_j\lambda_j$. 또한 $\min_j X_j\sim\text{Exp}(\sum_j\lambda_j)$이고, 최솟값의 크기와 승자 인덱스는 독립이다.

</details>

**Q4 (연결 고리).** (00b) 표본 $N$개의 MC 평균의 표준오차(SE)와 95% 신뢰구간은? 오차를 4배 줄이려면 $N$을 몇 배로?

<details><summary>정답</summary>

$\text{SE}=s/\sqrt N$, CI $=\bar x\pm1.96\,\text{SE}$. 오차 $\propto N^{-1/2}$이므로 16배. 이 노트북에서 '시각 $n$의 분포'를 추정할 때 SE는 베르누이 비율의 $\sqrt{p(1-p)/N}$이다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의 (Durrett 3e §1.1, Lawler 2e §1.1)

- **마르코프 사슬 (Markov chain).** 가산 상태공간 $S$ 위의 확률과정 $(X_n)_{n\ge0}$이 모든 $n$, 모든 상태 $x_0,\dots,x_{n-1},x,y$에 대해
  $$P(X_{n+1}=y\mid X_n=x,X_{n-1}=x_{n-1},\dots,X_0=x_0)=P(X_{n+1}=y\mid X_n=x)$$
  를 만족하면 **마르코프 성질(Markov property)** 을 가진다. 한 줄로: *미래는 현재만 보고 결정되고 과거는 잊는다.* (조건이 되는 사건의 확률이 0이면 그 식은 요구하지 않는다.)
- **시간 동질 (time-homogeneous).** $P(X_{n+1}=y\mid X_n=x)=p(x,y)$가 $n$에 의존하지 않을 때. 이 과정에서는 항상 시간 동질을 가정한다.
- **전이행렬 (transition matrix)** $P=(p(x,y))_{x,y\in S}$. 모든 원소 $\ge0$, 모든 행의 합 $\sum_y p(x,y)=1$ — **행확률행렬(row-stochastic)**. 코드에서는 상태를 $0,\dots,n-1$로 번호 매기고 `P[x, y]`로 쓴다. 행 $x$는 "$x$에서 출발한 다음 걸음의 분포"다.
- **초기분포** $\mu_0(x)=P(X_0=x)$. 시각 $n$의 분포는 **행벡터** $\mu_n=\mu_0P^n$ (행벡터 × 행렬 순서에 주의; 정리 C).
- **$n$-단계 전이확률** $p^n(x,y)=P(X_n=y\mid X_0=x)$. 정리 B에 의해 $p^n(x,y)=(P^n)_{xy}$, 즉 행렬 $n$제곱의 원소.
- **흡수 상태 (absorbing state).** $p(x,x)=1$인 상태. 들어가면 영원히 나오지 못한다. 도박꾼 파산 사슬에서 $0$과 $N$.

이 노트북의 두 예제:

| 이름 | 상태 | 전이행렬 |
|---|---|---|
| 날씨 사슬 (weather chain) | $0=$ 맑음, $1=$ 비 | $P=\begin{pmatrix}0.7&0.3\\0.4&0.6\end{pmatrix}$ |
| 도박꾼 파산 사슬 (gambler's ruin) | $\{0,1,\dots,N\}$ | $p(i,i+1)=p$, $p(i,i-1)=q$ ($0<i<N$), $p(0,0)=p(N,N)=1$ |

00a의 랜덤 워크 $i+S_n$을 $0$과 $N$에서 멈추게 한 것이 파산 사슬이다. $S_n$은 $S_{n-1}$과 새 동전 하나로 결정되므로 마르코프 성질은 자동이고, 경계를 흡수 상태로 두면 유한 상태 사슬이 된다.

### 2.2 정리 A — 유한차원 분포 공식 (경로 확률; Norris §1.1 Theorem 1.1.1, Durrett 3e §1.1)

> $$P(X_0=x_0,X_1=x_1,\dots,X_n=x_n)=\mu_0(x_0)\,p(x_0,x_1)\,p(x_1,x_2)\cdots p(x_{n-1},x_n).$$

**가정이 왜 필요한가**
- **마르코프 성질.** 곱셈 법칙 $P(A_0\cap\dots\cap A_n)=P(A_0)\prod_k P(A_k\mid A_0\cap\dots\cap A_{k-1})$은 아무 과정에나 성립한다. 각 인자를 $p(x_{k-1},x_k)$ 하나로 줄이려면 조건에서 $X_0,\dots,X_{k-2}$를 떼어낼 수 있어야 — 즉 과거를 잊어야 한다.
- **시간 동질.** 각 인자가 *같은* 행렬 $P$의 원소가 되려면 필요하다. 아니면 $p_k(x_{k-1},x_k)$처럼 시각마다 다른 행렬의 곱이 된다.

<details><summary>증명</summary>

$n$에 대한 귀납법. $n=0$이면 좌변은 $P(X_0=x_0)=\mu_0(x_0)$으로 정의 그대로다.

$n$에서 성립한다고 하자. 사건 $B=\{X_0=x_0,\dots,X_n=x_n\}$에 대해 곱셈 법칙으로
$$P(B\cap\{X_{n+1}=x_{n+1}\})=P(B)\cdot P(X_{n+1}=x_{n+1}\mid B).$$
$P(B)>0$이면 마르코프 성질로 $P(X_{n+1}=x_{n+1}\mid B)=P(X_{n+1}=x_{n+1}\mid X_n=x_n)$이고, 시간 동질로 이것은 $p(x_n,x_{n+1})$이다. 귀납 가정을 $P(B)$에 넣으면 원하는 식. $P(B)=0$이면 좌변도 0이고, 귀납 가정에 의해 우변의 앞부분 $\mu_0(x_0)\prod_{k\le n}p(x_{k-1},x_k)=P(B)=0$이므로 우변도 0. $\square$

</details>

### 2.3 정리 B — 채프먼-콜모고로프 방정식 (Chapman-Kolmogorov; Durrett 3e §1.2 Theorem 1.1, Ross 12e §4.2)

> 모든 $m,n\ge0$과 $x,y\in S$에 대해
> $$p^{m+n}(x,y)=\sum_{z\in S}p^m(x,z)\,p^n(z,y).$$
> 행렬로 쓰면 $P^{(m+n)}=P^{(m)}P^{(n)}$ (여기서 $P^{(k)}=(p^k(x,y))$), 따라서 $p^n(x,y)=(P^n)_{xy}$ — **행렬 $n$제곱**.

**가정이 왜 필요한가**
- **마르코프 성질.** "시각 $m$에 $z$에 있다"는 조건 아래 그 이후의 움직임이 시각 $0$부터 $m$까지의 경로(특히 $X_0=x$)와 독립이어야 $p^m(x,z)\,p^n(z,y)$로 인수분해된다.
- **시간 동질.** 시각 $m$부터 $n$단계 가는 확률이 시각 $0$부터 $n$단계 가는 확률 $p^n(z,y)$와 같아야 한다. 아니면 $P^{(m+n)}=P_1P_2\cdots P_{m+n}$ 꼴의 서로 다른 행렬의 곱이다.
- **가산 상태공간.** 중간 상태 $z$에 대한 합이 전확률 공식으로 정당화되려면 $\{X_m=z\}_{z\in S}$가 가산 분할이어야 한다.

<details><summary>증명</summary>

$P(X_0=x)>0$이라 하자. 사건 $\{X_m=z\}_{z\in S}$는 표본공간의 가산 분할이므로 전확률 공식으로
$$p^{m+n}(x,y)=P(X_{m+n}=y\mid X_0=x)=\sum_{z}P(X_{m+n}=y,\;X_m=z\mid X_0=x).$$
각 항을 곱셈 법칙으로 쪼개면 ($P(X_m=z,X_0=x)>0$인 $z$만 남기고, 나머지 항은 0)
$$P(X_{m+n}=y,X_m=z\mid X_0=x)=\underbrace{P(X_m=z\mid X_0=x)}_{=p^m(x,z)}\cdot P(X_{m+n}=y\mid X_m=z,X_0=x).$$
마지막 인자에서 $X_0=x$ 조건을 떼어야 한다. 정리 A를 시각 $0,\dots,m+n$ 전체에 적용하면, 경로 $x=x_0,x_1,\dots,x_m=z,\dots,x_{m+n}=y$의 확률은 $\mu_0(x_0)\prod_{k=1}^{m}p(x_{k-1},x_k)\cdot\prod_{k=m+1}^{m+n}p(x_{k-1},x_k)$로 **시각 $m$ 앞뒤가 곱으로 갈라진다.** 앞부분을 $x_1,\dots,x_{m-1}$에 대해, 뒷부분을 $x_{m+1},\dots,x_{m+n-1}$에 대해 더하면
$$P(X_0=x,X_m=z,X_{m+n}=y)=P(X_0=x,X_m=z)\cdot\sum_{x_{m+1},\dots,x_{m+n-1}}\prod_{k=m+1}^{m+n}p(x_{k-1},x_k),$$
즉 $P(X_{m+n}=y\mid X_m=z,X_0=x)$는 뒤쪽 합과 같고 $x$에 의존하지 않는다. 같은 계산을 $m=0$에서 하면 이 뒤쪽 합은 정확히 $P(X_n=y\mid X_0=z)=p^n(z,y)$이다(시간 동질: 어느 시각에서 시작하든 같은 $P$의 원소를 곱한다). 따라서
$$p^{m+n}(x,y)=\sum_z p^m(x,z)\,p^n(z,y).$$
우변은 행렬 곱의 $(x,y)$ 원소 정의이므로 $P^{(m+n)}=P^{(m)}P^{(n)}$. $P^{(1)}=P$, $P^{(0)}=I$이고, $P^{(n+1)}=P^{(n)}P^{(1)}=P^{(n)}P$이므로 귀납법으로 $P^{(n)}=P^n$. $\square$

</details>

### 2.4 정리 C — 시각 $n$의 분포 (Ross 12e §4.2, Durrett 3e §1.2)

> $\mu_n=\mu_0P^n$, 즉 $P(X_n=y)=\sum_x\mu_0(x)\,p^n(x,y)$.

**가정이 왜 필요한가**
- 전확률 공식만 필요하다. 정리 B가 $p^n(x,y)=(P^n)_{xy}$를 보장하므로 남는 것은 관례뿐: 분포는 **행벡터**이고 행렬을 **오른쪽**에 곱한다.

<details><summary>증명</summary>

$\{X_0=x\}_{x\in S}$로 분할하면 $P(X_n=y)=\sum_x P(X_0=x)\,P(X_n=y\mid X_0=x)=\sum_x\mu_0(x)\,p^n(x,y)=(\mu_0P^n)_y$. $\square$

</details>

### 2.5 직관

전이행렬은 '한 걸음의 규칙'이고 $P^n$은 '$n$걸음의 규칙'이다. 채프먼-콜모고로프는 $m+n$걸음을 $m$걸음 + $n$걸음으로 끊을 때 **중간 지점 $z$를 모두 더하라**는 것 — 경로 적분의 이산판이다. 행렬 곱 $(AB)_{xy}=\sum_z A_{xz}B_{zy}$의 정의 자체가 "중간 지점을 모두 더한다"이므로, 마르코프 사슬의 $n$걸음 확률이 행렬 거듭제곱인 것은 우연이 아니라 행렬 곱이 정확히 그 연산이기 때문이다.

00a의 랜덤 워크는 $S_n$이 $S_{n-1}$과 새 동전 하나로 결정되므로 마르코프 사슬이고, 흡수 경계를 두면 유한 상태 사슬이 된다. 이때 $P^n$의 $5$번 행은 '시작 자본 5로 시각 $n$에 어디에 있는가'의 분포이고, $n\to\infty$에서 두 흡수 상태 $\{0,N\}$에만 질량이 남는다. 그 극한의 $0$번 원소가 00a의 파산확률 $1-h(5)$이다. 반면 날씨 사슬처럼 어디서든 어디로든 갈 수 있으면 $P^n$의 **모든 행이 같은 벡터로** 수렴한다 — 시작 상태를 잊는다(01d, 01e의 주제).

### 2.6 함정(traps)

1. $P^n$은 행렬 거듭제곱 `np.linalg.matrix_power(P, n)`이지 원소별 거듭제곱 `P**n`이 아니다. `P**n`은 행합이 1이 아니다(4.2절에서 확인).
2. 분포는 행벡터: $\mu_n=\mu_0P^n$ (`mu @ matrix_power(P, n)`). 열벡터로 `P @ mu`를 하면 다른(틀린) 값이 나온다 — 그것은 $P$의 각 행에 대한 $\mu$-가중 평균이지 분포가 아니다.
3. 마르코프 사슬의 함수 $f(X_n)$은 일반적으로 마르코프가 아니다(E3). '현재 상태'에 충분한 정보를 넣어야 한다.
4. $p^n(x,y)>0$ 여부는 그래프에 길이 $n$ 경로가 있는지와 같지만, $p^n(x,x)$는 $n$에 따라 0과 양수를 오갈 수 있다(주기성, 01b). 파산 사슬 내부 상태가 그렇다: 홀수 걸음에 제자리로 돌아올 수 없다.
5. 시뮬레이션에서 '시각 $n$의 분포'를 추정하려면 경로 하나의 시간 평균이 아니라 **독립 경로 여러 개의 시각 $n$ 단면** `paths[:, n]`을 써야 한다. 시간 평균은 다른 것(정상분포, 01d)을 추정한다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — (1) 마르코프 성질, (2) 채프먼-콜모고로프 방정식을 각각 두 문장으로, (3) '왜 $P^n$이 행렬 곱이지 원소별 거듭제곱이 아닌가'를 한 문장으로 쓴 뒤 위 정의와 대조하세요.)

## 3. Predict — 코드를 돌리기 전에 예측

아래 셀의 `None`을 숫자로 바꿔 보세요. 첫 두 개는 손으로 계산할 수 있고, 나머지는 00a의 파산확률(시작 5, $N=10$, $p=0.45$)을 떠올리면 자릿수를 맞힐 수 있습니다. 5절의 비교표에서 이 값이 '내 예측' 열에 들어갑니다.

In [ ]:
predictions = {
    # 날씨 사슬 P = [[0.7, 0.3], [0.4, 0.6]] (0=맑음, 1=비)에서 오늘 맑을 때 모레도 맑을 확률 p^2(0,0)은?
    "p2_sunny_sunny": None,
    # 같은 사슬에서 p^10(0,0)은? (힌트: P^n의 두 행이 거의 같아진다. 0.7? 0.6? 0.5?)
    "p10_sunny_sunny": None,
    # 파산 사슬 N=10, p=0.45(이길 확률), 시작 5에서 300걸음 뒤 상태 0에 있을 확률 p^300(5,0)은? (00a의 파산확률과 비교)
    "ruin_from_5": None,
    # 같은 사슬에서 50걸음 뒤 상태 0에 있을 확률 p^50(5,0)은? (300걸음 값보다 클까 작을까?)
    "absorbed_at_0_by_50": None,
    # 50걸음 뒤에도 아직 게임 중(상태 1..9)일 확률은? (00a: E_5[T]는 약 25걸음이었다)
    "still_playing_50": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 날씨 사슬: 시각 $n$의 단면 vs $P^n$의 행

`simulate_chain`은 경로 `N_PATHS`개를 동시에 굴린다(시간에 대해서만 루프). 시각 $n$의 분포는 열 `paths[:, n]`의 도수다(함정 5).

In [ ]:
P_weather = np.array([[0.7, 0.3],
                      [0.4, 0.6]])               # 0 = Sunny, 1 = Rainy
assert is_stochastic(P_weather), "행합이 1이 아니다"
print("row sums:", P_weather.sum(axis=1))

N_STEPS_W = 10
paths = simulate_chain(P_weather, 0, N_STEPS_W, rng, n_paths=N_PATHS)   # shape (N_PATHS, 11), 모두 맑음에서 출발
print("paths shape:", paths.shape, "| first path:", paths[0])

# 각 시각 n의 경험분포 (열 단면의 도수) 와 정확한 P^n[0, :]
emp_dist = np.array([np.bincount(paths[:, n], minlength=2) / N_PATHS for n in range(N_STEPS_W + 1)])
exact_dist = np.array([matrix_power(P_weather, n)[0] for n in range(N_STEPS_W + 1)])

est_p2 = mc_proportion(paths[:, 2] == 0)
est_p10 = mc_proportion(paths[:, 10] == 0)
print("p^2(0,0):  MC", est_p2, " exact", matrix_power(P_weather, 2)[0, 0])
print("p^10(0,0): MC", est_p10, " exact", matrix_power(P_weather, 10)[0, 0])

In [ ]:
fig, ax = plt.subplots()
n_grid = np.arange(N_STEPS_W + 1)
w = 0.38
ax.bar(n_grid - w / 2, exact_dist[:, 0], width=w, label="exact  $P^n[0,0]$ (matrix_power)")
ax.bar(n_grid + w / 2, emp_dist[:, 0], width=w, label=f"MC  fraction of {N_PATHS} paths in Sunny")
ax.axhline(4 / 7, color="k", ls="--", lw=1, label="4/7 (limit, see 4.4)")
ax.set_xlabel("n (days)"); ax.set_ylabel("P(X_n = Sunny | X_0 = Sunny)")
ax.set_xticks(n_grid); ax.set_ylim(0.5, 1.02)
ax.set_title("Weather chain: P(X_n = Sunny | X_0 = Sunny), exact vs MC")
ax.legend(loc="upper right");

$n=0$에서 1(모두 맑음에서 출발), $n=1$에서 0.7, 그 뒤로는 빠르게 $4/7\approx0.571$로 내려앉는다. MC 막대의 오차는 $\sqrt{p(1-p)/N}\approx0.011$(FAST) 또는 $0.0035$(full) 규모다.

### 4.2 채프먼-콜모고로프를 숫자로 — 그리고 함정 1

In [ ]:
P7 = matrix_power(P_weather, 7)
ck_ok = np.allclose(P7, matrix_power(P_weather, 3) @ matrix_power(P_weather, 4))
print("P^7 == P^3 @ P^4 ?", ck_ok)
assert ck_ok, "채프먼-콜모고로프가 깨졌다"

print("matrix_power(P, 7) row sums:", matrix_power(P_weather, 7).sum(axis=1))
print("P**7 (elementwise!) row sums:", (P_weather**7).sum(axis=1), " <- not a stochastic matrix")

이제 $P^n$의 두 행의 차이 $\|P^n[0,:]-P^n[1,:]\|_1$을 본다. 두 행은 각각 '맑음에서 출발' / '비에서 출발'했을 때 시각 $n$의 분포이므로, 이 차이는 사슬이 **시작 상태를 얼마나 기억하는가**의 척도다.

In [ ]:
N_MAX = 15
n_arr = np.arange(1, N_MAX + 1)
row_gap = np.array([np.abs(matrix_power(P_weather, n)[0] - matrix_power(P_weather, n)[1]).sum() for n in n_arr])
lam2 = np.sort(np.linalg.eigvals(P_weather).real)[0]        # 두 번째 고유값 (= 1 - 0.3 - 0.4 = 0.3)
print("eigenvalues of P:", np.linalg.eigvals(P_weather).real, "| lambda_2 =", lam2)

fig, ax = plt.subplots()
ax.semilogy(n_arr, row_gap, "o-", label=r"$\|P^n[0,:]-P^n[1,:]\|_1$")
ax.semilogy(n_arr, 0.6 * 0.3 ** (n_arr - 1), "k--", lw=1, label=r"reference $0.6\cdot0.3^{\,n-1}$")
ax.set_xlabel("n"); ax.set_ylabel("L1 distance between rows")
ax.set_title("Rows of P^n converge to each other (rate 0.3)")
ax.legend();

기울기가 정확히 $\log 0.3$ — $P$의 두 번째 고유값 $\lambda_2=1-0.3-0.4=0.3$의 로그다(E2에서 2-상태 사슬에 대해 직접 유도한다). 일반적인 이유는 01e에서 다룬다. 여기서는 "행들이 기하급수적으로 같아진다"는 현상만 눈에 담아 두자.

### 4.3 도박꾼 파산 사슬: $P^n$의 행에서 파산확률 읽기

`gamblers_ruin_matrix(10, 0.45)`는 상태 $\{0,\dots,10\}$, $0$과 $10$이 흡수 상태인 $11\times11$ 행렬이다. 00a에서는 파산확률을 (i) 경로를 끝까지 굴려 세거나 (ii) 차분방정식을 풀어 얻었다. 이번엔 (iii) $P^{300}$의 $(5,0)$ 원소로 읽는다.

In [ ]:
N_GR, p_GR, x0_GR = 10, 0.45, 5
P_gr = gamblers_ruin_matrix(N_GR, p_GR)
assert is_stochastic(P_gr)
print("P_gr shape", P_gr.shape, "| absorbing rows:", P_gr[0, 0], P_gr[N_GR, N_GR], "| row 5:", P_gr[5])

N_STEPS_GR = 300
paths_gr = simulate_chain(P_gr, x0_GR, N_STEPS_GR, rng, n_paths=N_PATHS)   # shape (N_PATHS, 301)

# (a) 시각 300의 단면: 상태 0에 있는 비율  (vs p^300(5,0))
est_ruin300 = mc_proportion(paths_gr[:, 300] == 0)
exact_ruin300 = matrix_power(P_gr, 300)[x0_GR, 0]
# (b) 시각 50의 단면: {0}, {1..9}, {10} 세 비율
state50 = paths_gr[:, 50]
est_ruin50 = mc_proportion(state50 == 0)
est_play50 = mc_proportion((state50 >= 1) & (state50 <= 9))
est_win50 = mc_proportion(state50 == N_GR)
row50 = matrix_power(P_gr, 50)[x0_GR]
exact_ruin50, exact_play50, exact_win50 = row50[0], row50[1:N_GR].sum(), row50[N_GR]
print(f"t=300  ruined: MC {est_ruin300}  exact {exact_ruin300:.6f}")
print(f"t=50   ruined: MC {est_ruin50}  exact {exact_ruin50:.6f}")
print(f"t=50   playing: MC {est_play50}  exact {exact_play50:.6f}")
print(f"t=50   won:     MC {est_win50}  exact {exact_win50:.6f}")

In [ ]:
# exact P^n[5, :] grouped as {0}, {1..9}, {10} for n = 0..100  (P^n을 한 번씩만 곱해 누적)
N_AREA = 100
row = np.eye(N_GR + 1)[x0_GR]                       # mu_0 = e_5 (row vector)
mass = np.empty((N_AREA + 1, 3))
for n in range(N_AREA + 1):
    mass[n] = row[0], row[1:N_GR].sum(), row[N_GR]
    row = row @ P_gr                               # mu_{n+1} = mu_n P  (row vector on the left!)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
t_gr = np.arange(N_STEPS_GR + 1)
plot_paths(t_gr[:101], paths_gr[:, :101], ax=ax1, n_show=10, alpha=0.7)
for lev in (0, N_GR):
    ax1.axhline(lev, color="k", ls="--", lw=1)
ax1.set_ylabel("capital"); ax1.set_xlabel("n"); ax1.set_yticks(range(0, N_GR + 1, 2))
ax1.set_title("10 sample paths (absorbed paths go flat)")

ax2.stackplot(np.arange(N_AREA + 1), mass.T, labels=["ruined {0}", "still playing {1..9}", "won {10}"], alpha=0.8)
ax2.errorbar([50] * 3, [est_ruin50.mean, est_ruin50.mean + est_play50.mean, 1.0],
             yerr=[1.96 * est_ruin50.se, 1.96 * est_play50.se, 0], fmt="ko", ms=4, capsize=3, label="MC at n=50 (95% CI)")
ax2.set_xlabel("n"); ax2.set_ylabel("P^n[5, group]"); ax2.set_ylim(0, 1); ax2.legend(loc="center right")
ax2.set_title("Gambler's ruin N=10, p=0.45 from i=5: where is the chain at time n?")
fig.tight_layout()

왼쪽: 흡수되면 경로가 수평선이 된다 — $p(0,0)=p(10,10)=1$의 뜻. 오른쪽: '아직 게임 중'인 질량(가운데 띠)이 기하급수적으로 사라지고 $\{0\}$과 $\{10\}$이 남는다. $n=50$에서 MC 단면(검은 점)이 정확한 띠 경계와 겹친다.

### 4.4 수렴 예고: $P^n$의 행들은 어디로 가는가

In [ ]:
for n in [1, 2, 3, 5, 10, 20]:
    print(f"P_weather^{n:<2d} =", np.round(matrix_power(P_weather, n), 5).tolist())
print("limit rows should be (4/7, 3/7) =", (round(4 / 7, 5), round(3 / 7, 5)))

In [ ]:
P300 = matrix_power(P_gr, 300)
print("P_gr^300, rows 1..9 (columns 0 and 10; the middle 1..9 columns are ~0):")
for i in range(1, N_GR):
    print(f"  row {i}: [{P300[i, 0]:.4f}, ..., {P300[i, N_GR]:.4f}]   middle mass = {P300[i, 1:N_GR].sum():.1e}")
print("00a's h(i) = gamblers_ruin_prob(i, 10, 0.45):", [round(gamblers_ruin_prob(i, N_GR, p_GR), 4) for i in range(1, N_GR)])

날씨 사슬은 $n=10$이면 이미 두 행이 $(0.5714,0.4286)$로 같다 — 시작 상태를 잊는다. 파산 사슬은 $n=300$에서 각 행이 $(1-h_i,0,\dots,0,h_i)$ 꼴로 수렴하지만 **행마다 다르다**: 시작 자본을 영원히 기억한다. 그리고 $h_i$는 정확히 00a의 `gamblers_ruin_prob`이다. 왜 한쪽은 잊고 한쪽은 기억하는가 — 01b, 01c의 주제다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 `matrix_power`에서 나온다. 그중 두 개는 손 계산식과도 대조한다: $p^{10}(0,0)=\tfrac47+\tfrac37(0.3)^{10}$ (E2에서 유도), $p^{300}(5,0)\approx 1-h(5)$ (00a 정리 A; 300걸음 안에 흡수되지 않을 확률만큼 차이).

In [ ]:
exact_p2 = matrix_power(P_weather, 2)[0, 0]
exact_p10 = matrix_power(P_weather, 10)[0, 0]
np.testing.assert_allclose(exact_p2, 0.7 * 0.7 + 0.3 * 0.4)                       # 손 계산
np.testing.assert_allclose(exact_p10, 4 / 7 + (3 / 7) * 0.3**10)                   # E2의 닫힌 형식
not_absorbed_300 = matrix_power(P_gr, 300)[x0_GR, 1:N_GR].sum()
print(f"p^300(5,0) = {exact_ruin300:.7f} vs 1 - h(5) = {1 - gamblers_ruin_prob(x0_GR, N_GR, p_GR):.7f}"
      f"  (difference <= P(not absorbed by 300) = {not_absorbed_300:.1e})")

rows = [
    {"name": "p^2(Sunny, Sunny)", "predicted": predictions["p2_sunny_sunny"], "estimate": est_p2, "exact": exact_p2},
    {"name": "p^10(Sunny, Sunny)", "predicted": predictions["p10_sunny_sunny"], "estimate": est_p10, "exact": exact_p10},
    {"name": "p^300(5, 0)  ruin", "predicted": predictions["ruin_from_5"], "estimate": est_ruin300, "exact": exact_ruin300},
    {"name": "p^50(5, 0)  ruined by 50", "predicted": predictions["absorbed_at_0_by_50"], "estimate": est_ruin50, "exact": exact_ruin50},
    {"name": "sum_{y=1..9} p^50(5, y)  still playing", "predicted": predictions["still_playing_50"], "estimate": est_play50, "exact": exact_play50},
]
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **p^2(Sunny, Sunny)**: 어긋나면 `simulate_chain`이 행 `P[x, :]`가 아니라 열로 다음 상태를 뽑고 있거나, 단면 `paths[:, 2]` 대신 다른 열을 읽은 것이다.
- **p^10(Sunny, Sunny)**: 어긋나면 $P^{10}$을 `P**10`(원소별)으로 계산했거나(함정 1), 경로 길이가 10보다 짧다.
- **p^300(5, 0)**: 어긋나면 흡수 상태의 행 `P_gr[0]`이 $e_0$이 아니어서 흡수 뒤에도 움직인 것이다. 00a의 파산확률(0.7317)과 $10^{-7}$ 차이 이내여야 한다.
- **p^50(5, 0)**: 어긋나면 사슬의 $p$/$q$ 방향(위로 0.45)이 뒤집힌 것이다 — 뒤집히면 이 값이 0.25 근처로 떨어진다.
- **still playing at 50**: 어긋나면 상태 $1..9$ 마스크의 경계(`>= 1`, `<= 9`)를 잘못 잡은 것이다. 이 행은 세 비율의 합이 1이라는 것과도 묶여 있다.

표의 '내 예측' 중 3 SE 이상 어긋난 항목 하나를 골라 `log/mistakes.md`에 한 줄(날짜 | 01a | 내 예측 | 진실 | 왜 틀렸나)을 추가하세요.

In [ ]:
assert is_stochastic(P_weather) and is_stochastic(P_gr)
assert np.allclose(matrix_power(P_gr, 7), matrix_power(P_gr, 3) @ matrix_power(P_gr, 4))   # Chapman-Kolmogorov exact
assert_close(est_p2, exact_p2, k=4, name="p2_sunny_sunny")
assert_close(est_p10, exact_p10, k=4, name="p10_sunny_sunny")
assert_close(est_ruin300, exact_ruin300, k=4, name="ruin_from_5")
assert_close(est_ruin50, exact_ruin50, k=4, name="absorbed_at_0_by_50")
assert_close(est_play50, exact_play50, k=4, name="still_playing_50")
print("all checks passed")

## 6. 연습문제

### E1 계산

날씨 사슬 $P=\begin{pmatrix}0.7&0.3\\0.4&0.6\end{pmatrix}$에 대해 **손으로** (a) $P^2$를 구하고, (b) 오늘 맑을 때 모레 비 올 확률 $p^2(0,1)$, (c) 초기분포 $\mu_0=(1/2,1/2)$일 때 경로 확률 $P(X_0=\text{맑음},X_1=\text{맑음},X_2=\text{비})$를 구하시오. 그 뒤 코드로 확인하시오 — (c)는 정리 A의 식과, 경로 시뮬레이션의 도수 두 가지로.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $P^2=\begin{pmatrix}0.7\cdot0.7+0.3\cdot0.4 & 0.7\cdot0.3+0.3\cdot0.6\\ 0.4\cdot0.7+0.6\cdot0.4 & 0.4\cdot0.3+0.6\cdot0.6\end{pmatrix}=\begin{pmatrix}0.61&0.39\\0.52&0.48\end{pmatrix}$. 행합이 1인지 확인: $0.61+0.39=1$, $0.52+0.48=1$.

(b) $p^2(0,1)=0.7\cdot0.3+0.3\cdot0.6=0.21+0.18=0.39$ — "모레 비"는 "내일 맑고 모레 비" + "내일 비, 모레도 비"의 두 중간 지점 $z$를 더한 것(채프먼-콜모고로프).

(c) 정리 A: $\mu_0(0)\,p(0,0)\,p(0,1)=0.5\cdot0.7\cdot0.3=0.105$.

```python
P2 = matrix_power(P_weather, 2)
np.testing.assert_allclose(P2, [[0.61, 0.39], [0.52, 0.48]])
print("p^2(0,1) =", P2[0, 1])                                    # 0.39
mu0 = np.array([0.5, 0.5])
exact_path = mu0[0] * P_weather[0, 0] * P_weather[0, 1]           # 0.105
ex1 = simulate_chain(P_weather, mu0, 2, rng, n_paths=N_PATHS)     # 초기분포에서 출발
est_path = mc_proportion((ex1[:, 0] == 0) & (ex1[:, 1] == 0) & (ex1[:, 2] == 1))
print(est_path, "exact", exact_path)
assert_close(est_path, exact_path, k=4, name="path (S,S,R)")
print("mu_2 = mu_0 P^2 =", mu0 @ P2, " | wrong order P^2 @ mu_0 =", P2 @ mu0)   # 함정 2
```

함정 2의 확인: 올바른 $\mu_2=\mu_0P^2=(0.565,0.435)$인데, 순서를 바꾼 `P2 @ mu0`는 $(0.5,0.5)$ — 각 행의 합 1에 $1/2$을 곱한 값일 뿐 분포가 아니다. `mu0 = np.array([0.9, 0.1])`로 바꾸면 `mu0 @ P2`$=(0.601,0.399)$, `P2 @ mu0`$=(0.588,0.516)$로 후자는 합이 1도 아니다.

</details>

### E2 유도 후 시뮬레이션 검증

2-상태 사슬 $P=\begin{pmatrix}1-a&a\\b&1-b\end{pmatrix}$ ($0<a,b<1$)에 대해
$$p^n(0,0)=\frac{b}{a+b}+\frac{a}{a+b}(1-a-b)^n$$
임을 유도하시오. 날씨 사슬($a=0.3$, $b=0.4$)에서 $n=3$일 때 값을 구하고, `matrix_power`와 `simulate_chain`(시각 3의 단면, `N_PATHS`경로)으로 확인하시오. 4.2절의 기울기 $0.3$이 어디서 나오는지도 답하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** $u_n=p^n(0,0)$이라 하자. 채프먼-콜모고로프를 $m=n$, 마지막 한 걸음으로 쪼개면($P^{(n+1)}=P^{(n)}P$)
$$u_{n+1}=p^n(0,0)\,p(0,0)+p^n(0,1)\,p(1,0)=u_n(1-a)+(1-u_n)\,b=b+(1-a-b)\,u_n.$$
고정점 $u^*=b+(1-a-b)u^*\Rightarrow u^*=b/(a+b)$. 양변에서 빼면 $u_{n+1}-u^*=(1-a-b)(u_n-u^*)$, 따라서 $u_n-u^*=(1-a-b)^n(u_0-u^*)$. $u_0=p^0(0,0)=1$이므로 $u_0-u^*=a/(a+b)$:
$$p^n(0,0)=\frac{b}{a+b}+\frac{a}{a+b}(1-a-b)^n.\qquad\square$$

**날씨 사슬.** $a+b=0.7$, $1-a-b=0.3$: $p^3(0,0)=\tfrac47+\tfrac37(0.3)^3=0.571429+0.011571=0.5830$. (같은 식으로 $p^{10}(0,0)=\tfrac47+\tfrac37(0.3)^{10}=0.571431$ — 5절의 닫힌 형식.)

**기울기.** $P$의 고유값은 $1$과 $\lambda_2=1-a-b$ (대각합 $=2-a-b=1+\lambda_2$). $0<a,b<1$이면 $|\lambda_2|<1$이라 $u_n\to u^*$이고, 두 행의 차이 $p^n(0,0)-p^n(1,0)=(1-a-b)^n$이므로 $\|P^n[0,:]-P^n[1,:]\|_1=2\,|1-a-b|^n=0.6\cdot0.3^{n-1}$: 4.2절의 기준선 그대로다.

```python
a, b = 0.3, 0.4
closed = lambda n: b / (a + b) + a / (a + b) * (1 - a - b) ** n
print("closed form p^3(0,0) =", closed(3), "| matrix_power:", matrix_power(P_weather, 3)[0, 0])
np.testing.assert_allclose(closed(3), matrix_power(P_weather, 3)[0, 0])
np.testing.assert_allclose([closed(n) for n in range(16)], [matrix_power(P_weather, n)[0, 0] for n in range(16)])
ex2 = simulate_chain(P_weather, 0, 3, rng, n_paths=N_PATHS)
est_p3 = mc_proportion(ex2[:, 3] == 0)
print(est_p3, "SE ~", est_p3.se)
assert_close(est_p3, closed(3), k=4, name="p3_sunny_sunny")
print("eigenvalues:", np.linalg.eigvals(P_weather).real, "| 1 - a - b =", 1 - a - b)
```

</details>

### E3 가정을 깨보기

날씨 사슬 $X_n$에서 '최근 이틀의 비 온 날 수' $Y_n=X_{n-1}+X_n\in\{0,1,2\}$를 정의하자. $Y_n$은 마르코프 사슬인가? 정상 상태에서 $P(Y_{n+1}=2\mid Y_n=1,Y_{n-1}=0)$과 $P(Y_{n+1}=2\mid Y_n=1,Y_{n-1}=2)$를 계산하고, 길이 `scale(200_000)`의 경로 **하나**(`simulate_chain`)에서 조건부 도수로 확인하시오. 무조건부 $P(Y_{n+1}=2\mid Y_n=1)$도 구하시오. 마지막으로, 어떻게 상태를 바꾸면 다시 마르코프가 되는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**계산.** $Y_{n-1}=0$이면 $X_{n-2}=X_{n-1}=0$, 여기에 $Y_n=1$이면 $X_n=1$. 따라서 $Y_{n+1}=X_n+X_{n+1}=2$는 $X_{n+1}=1$과 같고, $X$의 마르코프 성질로
$$P(Y_{n+1}=2\mid Y_n=1,Y_{n-1}=0)=P(X_{n+1}=1\mid X_n=1)=0.6.$$
$Y_{n-1}=2$이면 $X_{n-1}=1$, 여기에 $Y_n=1$이면 $X_n=0$. 그러면 $Y_{n+1}=0+X_{n+1}\le1$이므로 $Y_{n+1}=2$는 **불가능**: 확률 $0$.

$0.6\ne0$ — 같은 현재 $Y_n=1$인데 과거 $Y_{n-1}$이 미래를 바꾸므로 $Y_n$은 마르코프가 아니다(함정 3). $Y_n=1$은 $(X_{n-1},X_n)=(0,1)$과 $(1,0)$ 두 경우를 뭉뚱그린 것이고, $Y_{n-1}$이 그 둘을 구별해 주기 때문이다.

**무조건부.** 정상분포 $\pi=(4/7,3/7)$(4.4절의 극한 행)에서 $P(X_{n-1}=0,X_n=1)=\tfrac47\cdot0.3=\tfrac{1.2}{7}$, $P(X_{n-1}=1,X_n=0)=\tfrac37\cdot0.4=\tfrac{1.2}{7}$로 같으므로 $Y_n=1$ 아래 두 경우의 비중은 반반이고 $P(Y_{n+1}=2\mid Y_n=1)=\tfrac12\cdot0.6+\tfrac12\cdot0=0.3$.

**고치는 법.** 상태를 쌍 $Z_n=(X_{n-1},X_n)\in\{0,1\}^2$으로 확장하면 $Z_{n+1}=(X_n,X_{n+1})$은 $Z_n$의 둘째 성분과 새 전이만으로 결정되므로 다시 마르코프다(4-상태 사슬). '현재 상태'에 충분한 정보를 넣는 것이 원칙이다.

```python
L = scale(200_000)
x = simulate_chain(P_weather, 0, L, rng)[0]         # 경로 하나 (길이 L+1); 시간 평균으로 정상 상태 근사
Y = x[:-1] + x[1:]                                  # Y_n = X_{n-1} + X_n
Y_prev, Y_cur, Y_next = Y[:-2], Y[1:-1], Y[2:]
def cond_freq(mask):
    return mc_proportion(Y_next[mask] == 2)
c0 = cond_freq((Y_cur == 1) & (Y_prev == 0))
c2 = cond_freq((Y_cur == 1) & (Y_prev == 2))
cu = cond_freq(Y_cur == 1)
print("P(Y'=2 | Y=1, Y_prev=0) =", c0, " exact 0.6")
print("P(Y'=2 | Y=1, Y_prev=2) =", c2, " exact 0.0")
print("P(Y'=2 | Y=1)           =", cu, " exact 0.3")
assert_close(c0, 0.6, k=4, name="Y_prev=0")
assert c2.mean == 0.0
assert_close(cu, 0.3, k=4, name="unconditional")
```

(경로 하나의 시간 평균을 쓴 것은 여기서 추정하려는 것이 *정상 상태*의 조건부 도수이기 때문이다 — 함정 5와 모순이 아니라, 무엇을 추정하느냐가 다르다. 한 경로 안의 표본은 독립이 아니므로 `mc_proportion`의 SE는 약간 과소평가되지만, 0.6 vs 0의 차이를 보기엔 충분하다.)

</details>

## 7. 정리

1. 마르코프 성질 = 미래는 현재 상태만 보고 결정된다; 시간 동질이면 행확률행렬 $P$ 하나가 사슬의 모든 것을 담는다.
2. 경로 확률은 $\mu_0(x_0)\prod_k p(x_{k-1},x_k)$ (정리 A); 시각 $n$의 분포는 행벡터 $\mu_n=\mu_0P^n$ (정리 C).
3. 채프먼-콜모고로프 $p^{m+n}(x,y)=\sum_z p^m(x,z)\,p^n(z,y)$ ⇔ $P^{(m+n)}=P^{(m)}P^{(n)}$ ⇔ $P^{(n)}=P^n$ — 행렬 거듭제곱이지 원소별 거듭제곱이 아니다.
4. 00a의 파산 랜덤 워크는 $\{0,\dots,N\}$ 위의 흡수 사슬이고, $P^{300}$의 $(5,0)$ 원소 $0.7317$이 00a의 파산확률과 $10^{-7}$ 이내로 같으며 시뮬레이션 단면과 4 SE 안에서 일치한다.
5. 날씨 사슬은 $P^n$의 모든 행이 $(4/7,3/7)$로 수렴하고(속도 $0.3^n$), 파산 사슬은 행마다 다른 극한 $(1-h_i,0,\dots,0,h_i)$ — 잊는 사슬과 기억하는 사슬의 차이가 01b·01c의 주제다.

**trap 카드**
- Q: `P**n`과 `matrix_power(P, n)` 중 $n$-단계 전이행렬은? 그리고 시각 $n$의 분포는 `P @ mu`인가 `mu @ P`인가?
- A: `matrix_power(P, n)` (행렬 곱). 분포는 행벡터이므로 `mu @ matrix_power(P, n)`. `P**n`은 원소별 거듭제곱이라 행합이 1이 아니고, `P @ mu`는 열벡터 관례라 다른 값이 나온다.

**다음 노트북: 01b** — 파산 사슬은 시작점마다 극한이 다르고 날씨 사슬은 같았다. 그 차이를 설명하는 언어(소통 클래스, 재귀/일시, 주기)가 01b의 상태 분류다.

log/progress.md 한 줄 템플릿:
`| YYYY-MM-DD | 01a 마르코프 사슬 정의·채프먼-콜모고로프 | __분 | 자기채점 _/5 | 헷갈린 것: ___ (예측 적중 _/5, 다음: 01b) |`